# 3x vs 3x1x variant counts

Load the combined TCR-peptide label table and count how many unique TCR variants are present only in the 3x1x population versus the 3x population.

In [ ]:
from pathlib import Path

import pandas as pd

csv_path = Path('/cluster/project/reddy/katja/NGS_pipeline/data/A3_Q30_combined_tcr_peptide_label_all_with_tcr_population.csv')
df = pd.read_csv(csv_path)

print(f'Loaded {df.shape[0]:,} rows and {df.shape[1]:,} columns')
display(df.head())

In [ ]:
df.columns.tolist()

In [ ]:
# Check the population labels available in the file.
df['tcr_population'].value_counts(dropna=False)

## Unique TCR variant counts

Here, a variant is counted as a unique value in the `tcr` column. The `tcr_population` column uses `pos3x1x` for variants only in 3x1x, `pos3x` for variants only in 3x, and `pos3x;pos3x1x` for variants found in both.

In [ ]:
variant_col = 'tcr'
population_col = 'tcr_population'

summary = pd.DataFrame(
    {
        'description': {
            'only_3x1x': 'unique TCR variants only in 3x1x',
            'only_3x': 'unique TCR variants only in 3x',
            'in_both_3x_and_3x1x': 'unique TCR variants in both 3x and 3x1x',
            'any_3x_data': 'unique TCR variants in 3x data, including overlap with 3x1x',
        },
        'population_label': {
            'only_3x1x': 'pos3x1x',
            'only_3x': 'pos3x',
            'in_both_3x_and_3x1x': 'pos3x;pos3x1x',
            'any_3x_data': 'pos3x or pos3x;pos3x1x',
        },
        'unique_tcr_variants': {
            'only_3x1x': df.loc[df[population_col].eq('pos3x1x'), variant_col].nunique(),
            'only_3x': df.loc[df[population_col].eq('pos3x'), variant_col].nunique(),
            'in_both_3x_and_3x1x': df.loc[df[population_col].eq('pos3x;pos3x1x'), variant_col].nunique(),
            'any_3x_data': df.loc[df[population_col].isin(['pos3x', 'pos3x;pos3x1x']), variant_col].nunique(),
        },
        'rows': {
            'only_3x1x': df[population_col].eq('pos3x1x').sum(),
            'only_3x': df[population_col].eq('pos3x').sum(),
            'in_both_3x_and_3x1x': df[population_col].eq('pos3x;pos3x1x').sum(),
            'any_3x_data': df[population_col].isin(['pos3x', 'pos3x;pos3x1x']).sum(),
        },
    }
)

summary

In [ ]:
print(f"Unique TCR variants only in 3x1x: {summary.loc['only_3x1x', 'unique_tcr_variants']:,}")
print(f"Unique TCR variants in 3x data including overlap: {summary.loc['any_3x_data', 'unique_tcr_variants']:,}")